In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet50, ResNet50_Weights, resnet18, ResNet18_Weights, mobilenet_v2, MobileNet_V2_Weights
import pandas as pd
import copy

# Configuration GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device utilisé : {device}")

# Hyperparamètres globaux (ajuste EPOCHS selon le temps disponible sur Colab)
BATCH_SIZE = 128
EPOCHS = 10
LR = 1e-3
IMG_SIZE = 128
NUM_CLASSES = 10

# Transformations (normalisation ImageNet)
transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_eval = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Téléchargement de CIFAR-10
full_train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_eval)

# Split Train (45 000) / Validation (5 000)
train_size = int(0.9 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

Device utilisé : cuda


100%|██████████| 170M/170M [19:08<00:00, 148kB/s]


Train: 45000 | Val: 5000 | Test: 10000


# ResNet50 et ResNet18

In [2]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

class ResNetWithFeatures(nn.Module):
    def __init__(self, arch="resnet50", pretrained=True, num_classes=10):
        super().__init__()
        if arch == "resnet50":
            weights = ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
            base = resnet50(weights=weights)
            self.feat_dim = 2048
        elif arch == "resnet18":
            weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
            base = resnet18(weights=weights)
            self.feat_dim = 512
        else:
            raise ValueError("Architecture non supportée")

        # Extraction du tronc convolutif avant avgpool
        self.features = nn.Sequential(*list(base.children())[:-2])
        self.avgpool = base.avgpool
        self.fc = nn.Linear(self.feat_dim, num_classes)

    def forward(self, x):
        fmap = self.features(x)
        pooled = torch.flatten(self.avgpool(fmap), 1)
        logits = self.fc(pooled)
        return logits, fmap

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits, _ = model(images)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return 100.0 * correct / total

# Fonctions d'entraînement

In [4]:
def train_standard(model, train_loader, val_loader, epochs=EPOCHS, lr=LR, name="Model"):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    best_acc = 0.0
    best_weights = copy.deepcopy(model.state_dict())

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            logits, _ = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        val_acc = evaluate(model, val_loader)
        if val_acc > best_acc:
            best_acc = val_acc
            best_weights = copy.deepcopy(model.state_dict())
        print(f"[{name}] Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(train_loader):.4f} | Val Acc: {val_acc:.2f}%")

    model.load_state_dict(best_weights)
    return model

def compute_distill_loss(s_logits, t_logits, metric="mse", T=4.0):
    if metric == "mse":
        return F.mse_loss(s_logits, t_logits)
    elif metric == "kl":
        return F.kl_div(
            F.log_softmax(s_logits / T, dim=1),
            F.softmax(t_logits / T, dim=1),
            reduction="batchmean"
        ) * (T * T)
    elif metric == "l1":
        return F.l1_loss(s_logits, t_logits)
    else:
        raise ValueError("Métrique inconnue")

def train_kd(student, teacher, train_loader, val_loader, epochs=EPOCHS, lr=LR,
             use_features=False, metric="mse", alpha=0.5, beta=0.5, name="Student KD"):
    student = student.to(device)
    teacher = teacher.to(device)

    # Gel des paramètres du Teacher
    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad = False

    # Adaptateur 1x1 si les dimensions des feature maps diffèrent
    adapter = None
    params = list(student.parameters())
    if use_features and student.feat_dim != teacher.feat_dim:
        adapter = nn.Conv2d(student.feat_dim, teacher.feat_dim, kernel_size=1).to(device)
        params += list(adapter.parameters())

    ce_loss = nn.CrossEntropyLoss()
    optimizer = optim.Adam(params, lr=lr)
    best_acc = 0.0
    best_weights = copy.deepcopy(student.state_dict())

    for epoch in range(epochs):
        student.train()
        if adapter:
            adapter.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            with torch.no_grad():
                t_logits, t_fmap = teacher(images)

            s_logits, s_fmap = student(images)

            loss_ce = ce_loss(s_logits, labels)
            loss_scores = compute_distill_loss(s_logits, t_logits, metric=metric)

            loss = (1 - alpha) * loss_ce + alpha * loss_scores

            if use_features:
                if adapter is not None:
                    s_fmap = adapter(s_fmap)
                # Aligne la taille spatiale si nécessaire (utile pour le bonus MobileNet)
                if s_fmap.shape[2:] != t_fmap.shape[2:]:
                    s_fmap = F.interpolate(s_fmap, size=t_fmap.shape[2:], mode='bilinear', align_corners=False)
                loss_feat = F.mse_loss(s_fmap, t_fmap)
                loss = loss + beta * loss_feat

            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        val_acc = evaluate(student, val_loader)
        if val_acc > best_acc:
            best_acc = val_acc
            best_weights = copy.deepcopy(student.state_dict())
        print(f"[{name}] Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(train_loader):.4f} | Val Acc: {val_acc:.2f}%")

    student.load_state_dict(best_weights)
    return student

# Entraînement du Teacher

In [5]:
results = {}

teacher = ResNetWithFeatures("resnet50", pretrained=True, num_classes=NUM_CLASSES)
total_p, _ = count_parameters(teacher)
print(f"Teacher (ResNet50) - Nombre de paramètres : {total_p:,}")

teacher = train_standard(teacher, train_loader, val_loader, epochs=EPOCHS, lr=1e-4, name="Teacher ResNet50")
teacher_test_acc = evaluate(teacher, test_loader)
print(f"--> Teacher Test Accuracy : {teacher_test_acc:.2f}%")

# Sauvegarde pour éviter de le ré-entraîner en cas de coupure Colab
torch.save(teacher.state_dict(), "teacher_resnet50.pth")
results["Teacher (ResNet50 Pretrained)"] = teacher_test_acc

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 221MB/s]


Teacher (ResNet50) - Nombre de paramètres : 23,528,522
[Teacher ResNet50] Epoch 1/10 | Loss: 0.2800 | Val Acc: 94.58%
[Teacher ResNet50] Epoch 2/10 | Loss: 0.1019 | Val Acc: 95.42%
[Teacher ResNet50] Epoch 3/10 | Loss: 0.0647 | Val Acc: 94.38%
[Teacher ResNet50] Epoch 4/10 | Loss: 0.0540 | Val Acc: 95.66%
[Teacher ResNet50] Epoch 5/10 | Loss: 0.0395 | Val Acc: 95.50%
[Teacher ResNet50] Epoch 6/10 | Loss: 0.0351 | Val Acc: 95.62%
[Teacher ResNet50] Epoch 7/10 | Loss: 0.0301 | Val Acc: 95.48%
[Teacher ResNet50] Epoch 8/10 | Loss: 0.0268 | Val Acc: 95.18%
[Teacher ResNet50] Epoch 9/10 | Loss: 0.0303 | Val Acc: 95.14%
[Teacher ResNet50] Epoch 10/10 | Loss: 0.0309 | Val Acc: 95.74%
--> Teacher Test Accuracy : 95.56%


# Stratégie 1: Knowledge Distillation avec un Student pré-entraîné

In [6]:
# Étape 2 : Student pré-entraîné seul
s1_base = ResNetWithFeatures("resnet18", pretrained=True, num_classes=NUM_CLASSES)
total_s, _ = count_parameters(s1_base)
print(f"Student (ResNet18) - Nombre de paramètres : {total_s:,}")

s1_base = train_standard(s1_base, train_loader, val_loader, epochs=EPOCHS, lr=1e-4, name="S1 - Base")
results["S1: Pretrained Base"] = evaluate(s1_base, test_loader)

# Étape 3 : Student pré-entraîné + KD Scores (MSE)
s1_kd_scores = ResNetWithFeatures("resnet18", pretrained=True, num_classes=NUM_CLASSES)
s1_kd_scores = train_kd(s1_kd_scores, teacher, train_loader, val_loader, epochs=EPOCHS, lr=1e-4,
                        use_features=False, metric="mse", name="S1 - KD Scores")
results["S1: Pretrained + KD Scores (MSE)"] = evaluate(s1_kd_scores, test_loader)

# Étape 4 : Student pré-entraîné + KD Scores & Feature Maps (MSE)
s1_kd_both = ResNetWithFeatures("resnet18", pretrained=True, num_classes=NUM_CLASSES)
s1_kd_both = train_kd(s1_kd_both, teacher, train_loader, val_loader, epochs=EPOCHS, lr=1e-4,
                      use_features=True, metric="mse", name="S1 - KD Scores+Features")
results["S1: Pretrained + KD Scores+Feat (MSE)"] = evaluate(s1_kd_both, test_loader)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 237MB/s]


Student (ResNet18) - Nombre de paramètres : 11,181,642
[S1 - Base] Epoch 1/10 | Loss: 0.3531 | Val Acc: 92.82%
[S1 - Base] Epoch 2/10 | Loss: 0.1263 | Val Acc: 93.84%
[S1 - Base] Epoch 3/10 | Loss: 0.0756 | Val Acc: 94.30%
[S1 - Base] Epoch 4/10 | Loss: 0.0480 | Val Acc: 94.18%
[S1 - Base] Epoch 5/10 | Loss: 0.0340 | Val Acc: 94.58%
[S1 - Base] Epoch 6/10 | Loss: 0.0288 | Val Acc: 94.46%
[S1 - Base] Epoch 7/10 | Loss: 0.0228 | Val Acc: 94.54%
[S1 - Base] Epoch 8/10 | Loss: 0.0216 | Val Acc: 94.38%
[S1 - Base] Epoch 9/10 | Loss: 0.0223 | Val Acc: 94.24%
[S1 - Base] Epoch 10/10 | Loss: 0.0173 | Val Acc: 94.76%
[S1 - KD Scores] Epoch 1/10 | Loss: 2.2766 | Val Acc: 93.54%
[S1 - KD Scores] Epoch 2/10 | Loss: 0.9193 | Val Acc: 94.94%
[S1 - KD Scores] Epoch 3/10 | Loss: 0.6835 | Val Acc: 94.92%
[S1 - KD Scores] Epoch 4/10 | Loss: 0.5647 | Val Acc: 95.42%
[S1 - KD Scores] Epoch 5/10 | Loss: 0.4791 | Val Acc: 95.12%
[S1 - KD Scores] Epoch 6/10 | Loss: 0.4245 | Val Acc: 95.58%
[S1 - KD Scores] E

# Stratégie 2: Knowledge Distillation avec un Student non-entraîné

In [7]:
# Étape 5 : Student from scratch seul
s2_base = ResNetWithFeatures("resnet18", pretrained=False, num_classes=NUM_CLASSES)
s2_base = train_standard(s2_base, train_loader, val_loader, epochs=EPOCHS, lr=1e-3, name="S2 - Scratch Base")
results["S2: Scratch Base"] = evaluate(s2_base, test_loader)

# Étape 6 : Student from scratch + KD Scores (MSE)
s2_kd_scores = ResNetWithFeatures("resnet18", pretrained=False, num_classes=NUM_CLASSES)
s2_kd_scores = train_kd(s2_kd_scores, teacher, train_loader, val_loader, epochs=EPOCHS, lr=1e-3,
                        use_features=False, metric="mse", name="S2 - Scratch KD Scores")
results["S2: Scratch + KD Scores (MSE)"] = evaluate(s2_kd_scores, test_loader)

# Étape 7 : Student from scratch + KD Scores & Feature Maps (MSE)
s2_kd_both = ResNetWithFeatures("resnet18", pretrained=False, num_classes=NUM_CLASSES)
s2_kd_both = train_kd(s2_kd_both, teacher, train_loader, val_loader, epochs=EPOCHS, lr=1e-3,
                      use_features=True, metric="mse", name="S2 - Scratch KD Scores+Feat")
results["S2: Scratch + KD Scores+Feat (MSE)"] = evaluate(s2_kd_both, test_loader)

[S2 - Scratch Base] Epoch 1/10 | Loss: 1.2922 | Val Acc: 55.76%
[S2 - Scratch Base] Epoch 2/10 | Loss: 0.8665 | Val Acc: 65.02%
[S2 - Scratch Base] Epoch 3/10 | Loss: 0.6656 | Val Acc: 67.28%
[S2 - Scratch Base] Epoch 4/10 | Loss: 0.5453 | Val Acc: 79.30%
[S2 - Scratch Base] Epoch 5/10 | Loss: 0.4614 | Val Acc: 80.08%
[S2 - Scratch Base] Epoch 6/10 | Loss: 0.3913 | Val Acc: 80.92%
[S2 - Scratch Base] Epoch 7/10 | Loss: 0.3364 | Val Acc: 81.20%
[S2 - Scratch Base] Epoch 8/10 | Loss: 0.2832 | Val Acc: 84.32%
[S2 - Scratch Base] Epoch 9/10 | Loss: 0.2403 | Val Acc: 83.08%
[S2 - Scratch Base] Epoch 10/10 | Loss: 0.2042 | Val Acc: 82.96%
[S2 - Scratch KD Scores] Epoch 1/10 | Loss: 6.8839 | Val Acc: 54.86%
[S2 - Scratch KD Scores] Epoch 2/10 | Loss: 4.3593 | Val Acc: 73.22%
[S2 - Scratch KD Scores] Epoch 3/10 | Loss: 3.3309 | Val Acc: 74.86%
[S2 - Scratch KD Scores] Epoch 4/10 | Loss: 2.7252 | Val Acc: 77.28%
[S2 - Scratch KD Scores] Epoch 5/10 | Loss: 2.3149 | Val Acc: 81.84%
[S2 - Scratch 

In [8]:
# Étape 9 : Comparaison avec d'autres métriques (KL Divergence et L1) sur le Student from scratch
for m in ["kl", "l1"]:
    s_metric = ResNetWithFeatures("resnet18", pretrained=False, num_classes=NUM_CLASSES)
    s_metric = train_kd(s_metric, teacher, train_loader, val_loader, epochs=EPOCHS, lr=1e-3,
                        use_features=False, metric=m, name=f"S2 - KD Scores ({m.upper()})")
    results[f"S2: Scratch + KD Scores ({m.upper()})"] = evaluate(s_metric, test_loader)

# Étape 8 : Tableau récapitulatif global
df_results = pd.DataFrame(list(results.items()), columns=["Expérience", "Test Accuracy (%)"])
display(df_results.sort_values(by="Test Accuracy (%)", ascending=False))

[S2 - KD Scores (KL)] Epoch 1/10 | Loss: 6.2411 | Val Acc: 54.92%
[S2 - KD Scores (KL)] Epoch 2/10 | Loss: 3.9468 | Val Acc: 72.02%
[S2 - KD Scores (KL)] Epoch 3/10 | Loss: 2.9065 | Val Acc: 75.48%
[S2 - KD Scores (KL)] Epoch 4/10 | Loss: 2.3672 | Val Acc: 79.54%
[S2 - KD Scores (KL)] Epoch 5/10 | Loss: 1.9985 | Val Acc: 81.32%
[S2 - KD Scores (KL)] Epoch 6/10 | Loss: 1.6669 | Val Acc: 82.54%
[S2 - KD Scores (KL)] Epoch 7/10 | Loss: 1.4435 | Val Acc: 84.12%
[S2 - KD Scores (KL)] Epoch 8/10 | Loss: 1.2302 | Val Acc: 83.78%
[S2 - KD Scores (KL)] Epoch 9/10 | Loss: 1.0797 | Val Acc: 83.30%
[S2 - KD Scores (KL)] Epoch 10/10 | Loss: 0.9151 | Val Acc: 86.42%
[S2 - KD Scores (L1)] Epoch 1/10 | Loss: 1.7545 | Val Acc: 59.98%
[S2 - KD Scores (L1)] Epoch 2/10 | Loss: 1.3683 | Val Acc: 74.56%
[S2 - KD Scores (L1)] Epoch 3/10 | Loss: 1.1742 | Val Acc: 75.78%
[S2 - KD Scores (L1)] Epoch 4/10 | Loss: 1.0432 | Val Acc: 78.08%
[S2 - KD Scores (L1)] Epoch 5/10 | Loss: 0.9395 | Val Acc: 77.92%
[S2 - KD 

,Expérience,Test Accuracy (%)
0,Teacher (ResNet50 Pretrained),95.56
2,S1: Pretrained + KD Scores (MSE),95.13
3,S1: Pretrained + KD Scores+Feat (MSE),95.12
1,S1: Pretrained Base,93.88
6,S2: Scratch + KD Scores+Feat (MSE),86.44
5,S2: Scratch + KD Scores (MSE),85.91
7,S2: Scratch + KD Scores (KL),85.65
8,S2: Scratch + KD Scores (L1),85.43
4,S2: Scratch Base,83.50


# CNN personnalisé

In [9]:
class CustomCNN(nn.Module):
    def __init__(self, div=1, num_classes=10):
        super().__init__()
        c1, c2, c3 = 64 // div, 128 // div, 256 // div
        self.feat_dim = c3
        self.features = nn.Sequential(
            nn.Conv2d(3, c1, kernel_size=3, padding=1), nn.BatchNorm2d(c1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(c1, c2, kernel_size=3, padding=1), nn.BatchNorm2d(c2), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(c2, c3, kernel_size=3, padding=1), nn.BatchNorm2d(c3), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(c3, num_classes)

    def forward(self, x):
        fmap = self.features(x)
        pooled = torch.flatten(self.avgpool(fmap), 1)
        logits = self.fc(pooled)
        return logits, fmap

custom_results = {}

# Custom Teacher (div=1)
custom_teacher = CustomCNN(div=1)
print(f"Custom Teacher params : {count_parameters(custom_teacher)[0]:,}")
custom_teacher = train_standard(custom_teacher, train_loader, val_loader, epochs=EPOCHS, lr=1e-3, name="Custom Teacher")
custom_results["Custom Teacher (/1)"] = evaluate(custom_teacher, test_loader)

# Custom Students compressés (/2 et /4)
for div in [2, 4]:
    # Sans KD
    s_custom = CustomCNN(div=div)
    print(f"Custom Student (/{div}) params : {count_parameters(s_custom)[0]:,}")
    s_custom = train_standard(s_custom, train_loader, val_loader, epochs=EPOCHS, lr=1e-3, name=f"Custom Student /{div} Base")
    custom_results[f"Custom Student (/{div}) Base"] = evaluate(s_custom, test_loader)

    # Avec KD (Scores + Features)
    s_custom_kd = CustomCNN(div=div)
    s_custom_kd = train_kd(s_custom_kd, custom_teacher, train_loader, val_loader, epochs=EPOCHS, lr=1e-3,
                           use_features=True, metric="mse", name=f"Custom Student /{div} KD")
    custom_results[f"Custom Student (/{div}) + KD"] = evaluate(s_custom_kd, test_loader)

display(pd.DataFrame(list(custom_results.items()), columns=["Modèle Custom", "Test Accuracy (%)"]))

Custom Teacher params : 374,282
[Custom Teacher] Epoch 1/10 | Loss: 1.6434 | Val Acc: 47.64%
[Custom Teacher] Epoch 2/10 | Loss: 1.3631 | Val Acc: 51.72%
[Custom Teacher] Epoch 3/10 | Loss: 1.2649 | Val Acc: 51.54%
[Custom Teacher] Epoch 4/10 | Loss: 1.2037 | Val Acc: 51.50%
[Custom Teacher] Epoch 5/10 | Loss: 1.1636 | Val Acc: 55.50%
[Custom Teacher] Epoch 6/10 | Loss: 1.1271 | Val Acc: 58.12%
[Custom Teacher] Epoch 7/10 | Loss: 1.0916 | Val Acc: 57.56%
[Custom Teacher] Epoch 8/10 | Loss: 1.0644 | Val Acc: 56.36%
[Custom Teacher] Epoch 9/10 | Loss: 1.0398 | Val Acc: 54.08%
[Custom Teacher] Epoch 10/10 | Loss: 1.0169 | Val Acc: 56.16%
Custom Student (/2) params : 94,986
[Custom Student /2 Base] Epoch 1/10 | Loss: 1.6545 | Val Acc: 40.08%
[Custom Student /2 Base] Epoch 2/10 | Loss: 1.3984 | Val Acc: 51.20%
[Custom Student /2 Base] Epoch 3/10 | Loss: 1.2984 | Val Acc: 49.64%
[Custom Student /2 Base] Epoch 4/10 | Loss: 1.2484 | Val Acc: 55.50%
[Custom Student /2 Base] Epoch 5/10 | Loss: 1

,Modèle Custom,Test Accuracy (%)
0,Custom Teacher (/1),58.66
1,Custom Student (/2) Base,59.08
2,Custom Student (/2) + KD,59.85
3,Custom Student (/4) Base,57.18
4,Custom Student (/4) + KD,56.57
